In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os


c:\Users\mrbil\OneDrive - Higher Education Commission\Desktop\LangGraph\LangGraph-Practical\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [10]:
load_dotenv(override=True)





True

In [ ]:
# model = ChatOpenAI()



# this is because the token os from github secrets and it is not working, so we need to test it immediately to see if it is working or not. If it is not working, then we need to check the token and the base url. If it is working, then we can proceed with the workflow.
model = ChatOpenAI(
    model="gpt-4o", 
    api_key=os.getenv("github_OPENAI_KEY"), 
    base_url="https://models.inference.ai.azure.com" 
)
 

# 3. Test it immediately
try:
    print("Testing Updated GitHub Token...")
    print("Response:", model.invoke("Hello!").content)
except Exception as e:
    print(f"Error: {e}")

Testing Updated GitHub Token...
Response: Hello! How can I assist you today? 😊


In [4]:
# create State
class LLMState(TypedDict):
    question_text: str
    answer_text: str

In [5]:
def llm_question_answer(state: LLMState) -> LLMState:
    
    # extract question
    question = state["question_text"]

    # make Prompt
    prompt = f"Answer the following question: {question}"

    # ask question to LLM
    model_response = model.invoke(prompt).content

    # update state with answer
    state["answer_text"] = model_response

    return state
    

In [6]:
# make the graph
llm_workflow = StateGraph(LLMState)

# add node
llm_workflow.add_node("llm_question_answer", llm_question_answer)

#add edge
llm_workflow.add_edge(START, "llm_question_answer")
llm_workflow.add_edge("llm_question_answer", END)

#compile the graph
llm_workflow = llm_workflow.compile()


In [9]:
# invoke the workflow with initial state
initial_state = {
    "question_text": "What is the capital of France?",
    
}   

final_state = llm_workflow.invoke(initial_state)
print(final_state["answer_text"])


The capital of France is **Paris**.
